In [1]:
import sys
from pathlib import Path

import duckdb
from IPython.display import display


working_directory = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        directory
        for directory in (working_directory, *working_directory.parents)
        if (directory / "src" / "data" / "loader.py").is_file()
        and (directory / "pyproject.toml").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError("Open the notebook from within the project directory.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.loader import load_customers

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Users\antek\Customer_Chunrn_End_to_End


In [2]:
if "connection" in globals():
    connection.close()

connection = duckdb.connect()

row_count = load_customers(connection)

print(f"Validated and loaded {row_count:,} customer rows.")

display(
    connection.sql("""
        SELECT *
        FROM raw_customers
        ORDER BY customer_id
        LIMIT 5
    """).df()
)

Validated and loaded 1,000,000 customer rows.


,customer_id,signup_date,age,gender,annual_income,education,marital_status,dependents,tenure,contract,...,has_streaming_tv,has_streaming_movies,customer_satisfaction,num_complaints,num_service_calls,late_payments,avg_monthly_gb,days_since_last_interaction,credit_score,churn
0,CUST0000000001,2022-12-12 12:53:58.199463,43,Female,55085.25,college,married,1,2,two_year,...,1,1,9.0,0.0,0,0,109.63,16,NaN,0
1,CUST0000000002,2022-01-13 12:53:58.199973,18,Male,60786.11,master,married,1,22,one_year,...,0,1,7.0,0.0,3,1,63.25,134,585.0,0
2,CUST0000000003,2023-09-04 12:53:58.199985,38,Female,73184.32,high_school,widowed,0,3,two_year,...,1,0,6.0,1.0,1,0,47.77,11,632.0,0
3,CUST0000000004,2022-06-27 12:53:58.199992,44,Male,40923.78,high_school,married,1,6,two_year,...,0,1,5.0,2.0,2,1,50.82,6,569.0,0
4,CUST0000000005,2022-12-08 12:53:58.199999,45,Female,36400.94,bachelor,single,0,9,two_year,...,0,1,8.0,1.0,1,0,16.74,18,657.0,0


In [3]:
categorical_columns = (
    "gender",
    "education",
    "marital_status",
    "contract",
    "payment_method",
)

for column in categorical_columns:
    print(f"\n{column.upper()}")

    display(
        connection.sql(f"""
            SELECT
                "{column}" AS category,
                COUNT(*) AS customers,
                ROUND(
                    100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
                    2
                ) AS percentage
            FROM raw_customers
            GROUP BY "{column}"
            ORDER BY customers DESC, category
        """).df()
    )


GENDER


,category,customers,percentage
0,Male,490166,49.02
1,Female,489595,48.96
2,Other,20239,2.02



EDUCATION


,category,customers,percentage
0,bachelor,300324,30.03
1,college,249610,24.96
2,high_school,249348,24.93
3,master,150468,15.05
4,phd,50250,5.03



MARITAL_STATUS


,category,customers,percentage
0,married,450115,45.01
1,single,350830,35.08
2,divorced,148976,14.90
3,widowed,50079,5.01



CONTRACT


,category,customers,percentage
0,one_year,550468,55.05
1,two_year,429540,42.95
2,month_to_month,19992,2.00



PAYMENT_METHOD


,category,customers,percentage
0,credit_card,349706,34.97
1,electronic_check,250346,25.03
2,bank_transfer,249910,24.99
3,mailed_check,150038,15.00


In [4]:
display(
    connection.sql("""
        SELECT
            MIN(signup_date) AS earliest_signup,
            MAX(signup_date) AS latest_signup,
            MIN(tenure) AS shortest_tenure_months,
            MAX(tenure) AS longest_tenure_months,
            MIN(
                signup_date + tenure * INTERVAL '1 month'
            ) AS earliest_implied_endpoint,
            MAX(
                signup_date + tenure * INTERVAL '1 month'
            ) AS latest_implied_endpoint
        FROM raw_customers
    """).df()
)

display(
    connection.sql("""
        SELECT
            customer_id,
            signup_date,
            tenure,
            signup_date + tenure * INTERVAL '1 month'
                AS implied_endpoint
        FROM raw_customers
        ORDER BY customer_id
        LIMIT 10
    """).df()
)

display(
    connection.sql("""
        SELECT
            DATE_TRUNC(
                'year',
                signup_date + tenure * INTERVAL '1 month'
            ) AS implied_endpoint_year,
            COUNT(*) AS customers
        FROM raw_customers
        GROUP BY implied_endpoint_year
        ORDER BY implied_endpoint_year
    """).df()
)

,earliest_signup,latest_signup,shortest_tenure_months,longest_tenure_months,earliest_implied_endpoint,latest_implied_endpoint
0,2021-01-13 12:53:58.205311,2026-01-11 12:54:09.480291,1,72,2021-02-13 12:53:59.761634,2032-01-11 12:54:09.468637


,customer_id,signup_date,tenure,implied_endpoint
0,CUST0000000001,2022-12-12 12:53:58.199463,2,2023-02-12 12:53:58.199463
1,CUST0000000002,2022-01-13 12:53:58.199973,22,2023-11-13 12:53:58.199973
2,CUST0000000003,2023-09-04 12:53:58.199985,3,2023-12-04 12:53:58.199985
3,CUST0000000004,2022-06-27 12:53:58.199992,6,2022-12-27 12:53:58.199992
4,CUST0000000005,2022-12-08 12:53:58.199999,9,2023-09-08 12:53:58.199999
5,CUST0000000006,2023-01-12 12:53:58.200006,47,2026-12-12 12:53:58.200006
6,CUST0000000007,2021-04-23 12:53:58.200012,26,2023-06-23 12:53:58.200012
7,CUST0000000008,2023-03-04 12:53:58.200018,18,2024-09-04 12:53:58.200018
8,CUST0000000009,2021-07-18 12:53:58.200023,1,2021-08-18 12:53:58.200023
9,CUST0000000010,2025-09-12 12:53:58.200029,31,2028-04-12 12:53:58.200029


,implied_endpoint_year,customers
0,2021-01-01,42308
1,2022-01-01,104295
2,2023-01-01,142321
3,2024-01-01,165116
4,2025-01-01,179708
5,2026-01-01,144747
6,2027-01-01,94868
7,2028-01-01,57575
8,2029-01-01,34884
9,2030-01-01,21242


In [5]:
count_columns = (
    "num_complaints",
    "num_service_calls",
    "late_payments",
    "num_services",
)

for column in count_columns:
    print(f"\n{column.upper()}")

    display(
        connection.sql(f"""
            SELECT
                COUNT("{column}") AS non_null_values,
                COUNT(*) FILTER (
                    WHERE "{column}" <> FLOOR("{column}")
                ) AS fractional_values,
                ROUND(
                    100.0 * COUNT(*) FILTER (
                        WHERE "{column}" <> FLOOR("{column}")
                    ) / NULLIF(COUNT("{column}"), 0),
                    4
                ) AS fractional_percentage
            FROM raw_customers
        """).df()
    )


NUM_COMPLAINTS


,non_null_values,fractional_values,fractional_percentage
0,970094,0,0.0



NUM_SERVICE_CALLS


,non_null_values,fractional_values,fractional_percentage
0,1000000,0,0.0



LATE_PAYMENTS


,non_null_values,fractional_values,fractional_percentage
0,1000000,0,0.0



NUM_SERVICES


,non_null_values,fractional_values,fractional_percentage
0,1000000,0,0.0


In [6]:
display(
    connection.sql("""
        WITH checks AS (
            SELECT
                *,
                has_phone_service
                    + has_internet_service
                    + has_online_security
                    + has_online_backup
                    + has_device_protection
                    + has_tech_support
                    + has_streaming_tv
                    + has_streaming_movies AS service_flag_sum
            FROM raw_customers
        )
        SELECT
            COUNT(*) AS customers,
            COUNT(*) FILTER (
                WHERE num_services <> service_flag_sum
            ) AS service_count_mismatches,
            ROUND(
                100.0 * COUNT(*) FILTER (
                    WHERE num_services <> service_flag_sum
                ) / COUNT(*),
                2
            ) AS mismatch_percentage,
            COUNT(*) FILTER (
                WHERE has_internet_service = 0
                  AND (
                      has_online_security = 1
                      OR has_online_backup = 1
                      OR has_device_protection = 1
                      OR has_tech_support = 1
                      OR has_streaming_tv = 1
                      OR has_streaming_movies = 1
                  )
            ) AS internet_addons_without_internet
        FROM checks
    """).df()
)

,customers,service_count_mismatches,mismatch_percentage,internet_addons_without_internet
0,1000000,879888,87.99,128780


In [7]:
display(
    connection.sql("""
        SELECT
            COUNT(*) AS customers,
            COUNT(*) FILTER (
                WHERE senior_citizen <>
                    CASE WHEN age >= 65 THEN 1 ELSE 0 END
            ) AS mismatches,
            ROUND(
                100.0 * COUNT(*) FILTER (
                    WHERE senior_citizen <>
                        CASE WHEN age >= 65 THEN 1 ELSE 0 END
                ) / COUNT(*),
                2
            ) AS mismatch_percentage
        FROM raw_customers
    """).df()
)

display(
    connection.sql("""
        SELECT
            age,
            senior_citizen,
            COUNT(*) AS customers
        FROM raw_customers
        WHERE senior_citizen <>
            CASE WHEN age >= 65 THEN 1 ELSE 0 END
        GROUP BY age, senior_citizen
        ORDER BY customers DESC
        LIMIT 10
    """).df()
)

,customers,mismatches,mismatch_percentage
0,1000000,254538,25.45


,age,senior_citizen,customers
0,65,0,8399
1,18,1,8335
2,66,0,7783
3,67,0,6928
4,68,0,6284
5,69,0,5664
6,44,1,5332
7,47,1,5277
8,42,1,5246
9,45,1,5190


In [8]:
display(
    connection.sql("""
        WITH billing AS (
            SELECT
                *,
                totalcharges
                    / NULLIF(monthlycharges * tenure, 0)
                    AS billing_ratio
            FROM raw_customers
        )
        SELECT
            COUNT(*) AS customers,
            COUNT(billing_ratio) AS comparable_rows,
            COUNT(*) FILTER (
                WHERE billing_ratio IS NULL
            ) AS undefined_ratios,
            COUNT(*) FILTER (
                WHERE billing_ratio < 0.9
                   OR billing_ratio > 1.1
            ) AS outside_tolerance,
            ROUND(
                100.0 * COUNT(*) FILTER (
                    WHERE billing_ratio < 0.9
                       OR billing_ratio > 1.1
                ) / NULLIF(COUNT(billing_ratio), 0),
                2
            ) AS outside_percentage,
            MIN(billing_ratio) AS minimum_ratio,
            MEDIAN(billing_ratio) AS median_ratio,
            MAX(billing_ratio) AS maximum_ratio
        FROM billing
    """).df()
)

,customers,comparable_rows,undefined_ratios,outside_tolerance,outside_percentage,minimum_ratio,median_ratio,maximum_ratio
0,1000000,1000000,0,157229,15.72,0.030939,0.982151,1.643833


In [9]:
documented_ranges = {
    "age": (18, 90),
    "dependents": (0, 5),
    "tenure": (1, 72),
    "monthlycharges": (20, 200),
    "num_services": (1, 6),
    "customer_satisfaction": (1, 10),
    "num_complaints": (0, 8),
    "num_service_calls": (0, 12),
    "late_payments": (0, 5),
    "days_since_last_interaction": (1, 365),
    "credit_score": (300, 850),
}

range_queries = []

for column, (lower, upper) in documented_ranges.items():
    range_queries.append(f"""
        SELECT
            '{column}' AS column_name,
            {lower} AS expected_min,
            {upper} AS expected_max,
            MIN("{column}") AS observed_min,
            MAX("{column}") AS observed_max,
            COUNT(*) FILTER (
                WHERE "{column}" IS NULL
            ) AS missing_values,
            COUNT(*) FILTER (
                WHERE "{column}" < {lower}
                   OR "{column}" > {upper}
            ) AS outside_range
        FROM raw_customers
    """)

range_results = connection.sql(
    " UNION ALL ".join(range_queries)
).df()

display(
    range_results.sort_values(
        ["outside_range", "column_name"],
        ascending=[False, True],
    ).reset_index(drop=True)
)

,column_name,expected_min,expected_max,observed_min,observed_max,missing_values,outside_range
0,monthlycharges,20,200,20.0,854.95673,0,979
1,age,18,90,18.0,90.00000,0,0
2,credit_score,300,850,300.0,850.00000,40395,0
3,customer_satisfaction,1,10,1.0,9.00000,19921,0
4,days_since_last_interaction,1,365,1.0,365.00000,0,0
5,dependents,0,5,0.0,5.00000,0,0
6,late_payments,0,5,0.0,5.00000,0,0
7,num_complaints,0,8,0.0,7.00000,29906,0
8,num_service_calls,0,12,0.0,12.00000,0,0
9,num_services,1,6,1.0,6.00000,0,0


In [10]:
addon_columns = (
    "has_online_security",
    "has_online_backup",
    "has_device_protection",
    "has_tech_support",
    "has_streaming_tv",
    "has_streaming_movies",
)

for column in addon_columns:
    print(f"\n{column.upper()}")

    display(
        connection.sql(f"""
            SELECT
                has_internet_service,
                COUNT(*) AS customers,
                SUM("{column}") AS addon_customers,
                ROUND(
                    100.0 * SUM("{column}") / COUNT(*),
                    2
                ) AS addon_percentage
            FROM raw_customers
            GROUP BY has_internet_service
            ORDER BY has_internet_service
        """).df()
    )


HAS_ONLINE_SECURITY


,has_internet_service,customers,addon_customers,addon_percentage
0,0,150258,0.0,0.00
1,1,849742,339468.0,39.95



HAS_ONLINE_BACKUP


,has_internet_service,customers,addon_customers,addon_percentage
0,0,150258,0.0,0.0
1,1,849742,425723.0,50.1



HAS_DEVICE_PROTECTION


,has_internet_service,customers,addon_customers,addon_percentage
0,0,150258,0.0,0.00
1,1,849742,297082.0,34.96


... (2 outputs left out)


HAS_STREAMING_TV


,has_internet_service,customers,addon_customers,addon_percentage
0,0,150258,90420.0,60.18
1,1,849742,509570.0,59.97



HAS_STREAMING_MOVIES


,has_internet_service,customers,addon_customers,addon_percentage
0,0,150258,82517.0,54.92
1,1,849742,466569.0,54.91


## Initial Data Verification

### Objective

Before exploring churn drivers or engineering features, we assess whether
the downloaded dataset matches its documentation and whether its fields
are internally consistent.

The dataset is synthetic.

### What is done in the code

1. Loaded the CSV through DuckDB loader.
2. Enforced expected column names and data types.
3. Checked customer IDs, required values, binary flags, and finite,
   nonnegative numeric values.
4. Inspected categorical distributions and missingness.
5. Compared signup dates with tenure and service counts with service flags.
6. Tested specific claims from the dataset description.

These checks are only diagnostic for now, no data processing is doen at this point

### Dataset Overview

- **Rows:** 1,000,000
- **Columns:** 32, including the customer identifier and churn target
- **Customer IDs:** unique, with no missing IDs
- **Observed churn rate:** 9.92%, rather than the documented 20-25%
- **Contracts:** 55.05% one-year, 42.95% two-year, and 2.00% month-to-month

The documentation's feature count and churn rate do not match the
downloaded file. All analysis will be done based on the caluclated statistics not the dataset description.

### Missing Values

| Column | Missing |
|---|---:|
| Average monthly data usage | 5.00% |
| Credit score | 4.04% |
| Annual income | 3.00% |
| Number of complaints | 2.99% |
| Customer satisfaction | 1.99% |

Missing values remain unchanged. In particular, missing complaints do not
mean zero complaints.

### Consistency Findings

#### 1. Signup Dates and Tenure

Signup dates span approximately January 2021 to January 2026. However,
adding tenure in months produces implied endpoints from February 2021
to January 2032.


#### 2. Senior-Citizen Flag

The documentation defines `senior_citizen` as 1 when age is at least 65.
However, **254,538 records (25.45%)** do not follow the definition.


#### 3. Billing Relationship

The documentation describes:

`totalcharges ≈ monthlycharges × tenure`, with ±10% noise.

However, **157,229 records (15.72%)** fall outside the interval.

- Median actual-to-expected ratio: **0.982**
- Minimum ratio: **0.031**
- Maximum ratio: **1.644**

This is a documentation discrepancy.

#### 4. Numeric Ranges and Count Fields

All checked non-null values fall within their documented ranges except
monthly charges:

- **979 records (0.0979%)** exceed $200.
- Maximum monthly charge: approximately **$854.96**.

The frequency is close to the documented 0.1% monetary-outlier rate.

The checked count fields contain no fractional values. Floating-point
storage of complaints therefore does not imply fractional complaints.

#### 5. Service Counts

For **879,888 records (87.99%)**, `num_services` differs from the sum of
the eight service flags.

The precise counting definition is unclear. Both will be kept as maybe they do not have to match, they might be different things.

#### 6. Internet Subscription and Add-ons

The  check flagged 128,780 customers without internet.
Breaking it down shows that the services behave differently:

- Online security, online backup, and device protection occur only
  among internet subscribers in this file.
- Tech support adoption is **20.04%** without internet versus **54.99%**
  with internet.
- Streaming TV adoption is approximately **60%** in both groups.
- Streaming movie adoption is approximately **55%** in both groups.


In [11]:
numeric_columns = [
    "age",
    "tenure",
    "monthlycharges",
    "totalcharges",
    "annual_income",
    "avg_monthly_gb",
    "customer_satisfaction",
    "num_complaints",
    "credit_score",
    "days_since_last_interaction",
]

selected_columns = ", ".join(f'"{column}"' for column in numeric_columns)

numeric_summary = connection.sql(f"""
    WITH numeric_values AS (
        UNPIVOT (
            SELECT {selected_columns}
            FROM raw_customers
        )
        ON COLUMNS(*)
        INTO NAME feature VALUE value
    )
    SELECT
        feature,
        COUNT(*) AS observed,
        MIN(value) AS minimum,
        QUANTILE_CONT(value, 0.01) AS p01,
        QUANTILE_CONT(value, 0.25) AS q1,
        MEDIAN(value) AS median,
        QUANTILE_CONT(value, 0.75) AS q3,
        QUANTILE_CONT(value, 0.99) AS p99,
        MAX(value) AS maximum,
        AVG(value) AS mean
    FROM numeric_values
    GROUP BY feature
    ORDER BY feature
""").df()

display(numeric_summary.round(2))

,feature,observed,minimum,p01,q1,median,q3,p99,maximum,mean
0,age,1000000,18.0,18.00,34.00,44.00,55.00,79.00,90.00,44.72
1,annual_income,970041,20000.0,20000.00,32712.56,48954.60,73475.13,197747.89,250000.00,58788.31
2,avg_monthly_gb,949988,0.0,0.00,12.84,27.77,51.23,209.05,557.82,39.10
3,credit_score,959605,300.0,470.00,619.00,680.00,740.00,850.00,850.00,678.56
4,customer_satisfaction,980079,1.0,1.00,4.00,7.00,8.00,9.00,9.00,6.16
5,days_since_last_interaction,1000000,1.0,1.00,12.00,31.00,62.00,207.00,365.00,44.49
6,monthlycharges,1000000,20.0,36.91,70.49,85.48,100.68,145.20,854.96,86.44
7,num_complaints,970094,0.0,0.00,0.00,1.00,1.00,3.00,7.00,0.70
8,tenure,1000000,1.0,1.00,6.00,16.00,33.00,72.00,72.00,22.38
9,totalcharges,1000000,16.3,53.67,484.53,1249.75,2617.66,7785.64,16252.86,1837.32


In [12]:
import plotly.express as px


bin_widths = {
    "age": 5,
    "tenure": 3,
    "monthlycharges": 10,
    "totalcharges": 250,
    "annual_income": 5000,
    "avg_monthly_gb": 10,
    "customer_satisfaction": 1,
    "num_complaints": 1,
    "credit_score": 25,
    "days_since_last_interaction": 15,
}

for column, width in bin_widths.items():
    distribution = connection.sql(f"""
        SELECT
            FLOOR("{column}" / {width}) * {width} AS bin_start,
            COUNT(*) AS customers
        FROM raw_customers
        WHERE "{column}" IS NOT NULL
        GROUP BY bin_start
        ORDER BY bin_start
    """).df()

    figure = px.bar(
        distribution,
        x="bin_start",
        y="customers",
        title=f"{column} — bin width: {width}",
        labels={
            "bin_start": f"{column} (bin start)",
            "customers": "Customers",
        },
    )
    figure.update_traces(width=width)
    figure.update_layout(bargap=0)
    figure.show()

In [13]:
missing_columns = [
    "annual_income",
    "customer_satisfaction",
    "num_complaints",
    "avg_monthly_gb",
    "credit_score",
]

missing_flags = ", ".join(
    f'"{column}" IS NULL AS "{column}_missing"'
    for column in missing_columns
)

missing_patterns = connection.sql(f"""
    SELECT
        {missing_flags},
        COUNT(*) AS customers,
        ROUND(
            100.0 * COUNT(*) / SUM(COUNT(*)) OVER (),
            2
        ) AS percentage
    FROM raw_customers
    GROUP BY ALL
    ORDER BY customers DESC
""").df()

display(missing_patterns)

,annual_income_missing,customer_satisfaction_missing,num_complaints_missing,avg_monthly_gb_missing,credit_score_missing,customers,percentage
0,False,False,False,False,False,840765,84.08
1,False,False,False,True,False,44302,4.43
2,False,False,False,False,True,35377,3.54
3,True,False,False,False,False,26008,2.60
4,False,False,True,False,False,25898,2.59
5,False,True,False,False,False,17051,1.71
6,False,False,False,True,True,1837,0.18
7,False,False,True,True,False,1394,0.14
8,True,False,False,True,False,1344,0.13
9,True,False,False,False,True,1122,0.11


In [ ]:
outlier_columns = {
    "monthlycharges",
    "totalcharges",
    "annual_income",
    "avg_monthly_gb",
}

for row in numeric_summary.itertuples(index=False):
    if row.feature not in outlier_columns:
        continue

    iqr = row.q3 - row.q1
    lower_bound = row.q1 - 1.5 * iqr
    upper_bound = row.q3 + 1.5 * iqr
    column = row.feature

    print(
        f"\n{column}: "
        f"lower={lower_bound:.2f}, upper={upper_bound:.2f}"
    )

    display(
        connection.sql(f"""
            SELECT
                COUNT("{column}") AS observed,
                COUNT(*) FILTER (
                    WHERE "{column}" < {lower_bound}
                ) AS below_lower_bound,
                COUNT(*) FILTER (
                    WHERE "{column}" > {upper_bound}
                ) AS above_upper_bound,
                ROUND(
                    100.0 * COUNT(*) FILTER (
                        WHERE "{column}" < {lower_bound}
                           OR "{column}" > {upper_bound}
                    ) / NULLIF(COUNT("{column}"), 0),
                    2
                ) AS flagged_percentage
            FROM raw_customers
        """).df()
    )


annual_income: lower=-28431.30, upper=134618.99


,observed,below_lower_bound,above_upper_bound,flagged_percentage
0,970041,0,44871,4.63



avg_monthly_gb: lower=-44.75, upper=108.81


,observed,below_lower_bound,above_upper_bound,flagged_percentage
0,949988,0,56428,5.94



monthlycharges: lower=25.20, upper=145.97


,observed,below_lower_bound,above_upper_bound,flagged_percentage
0,1000000,379,9361,0.97



totalcharges: lower=-2715.16, upper=5817.35


,observed,below_lower_bound,above_upper_bound,flagged_percentage
0,1000000,0,45654,4.57


## Numerical Distributions, Missingness, and Outlier Screening

### Distribution Findings

| Feature | Observation |
|---|---|
| Age | Concentrated around ages 40–50; median 44, observed range 18–90. |
| Tenure | Right-skewed; median 16 months, with upper limit of around 72 months. |
| Monthly charges | Approximately normal main body centered around $80–$90; median $85.48, with an upper tail reaching $854.96. |
| Total charges | Right-skewed; median $1,249.75 and maximum $16,252.86. |
| Annual income | Right-skewed; median $48,954.60, with  accumulation at $250,000. |
| Monthly data usage | Right-skewed; median 27.77 GB, with a long tail reaching 557.82 GB. |
| Satisfaction | Scores are concentrated toward the upper end; median 7, observed range 1–9. |
| Complaints | Concentrated at low counts; median 1, with values reaching 7. |
| Credit score | Centered around 680, with accumulation at the upper limit of 850. |
| Days since interaction | Right-skewed; median 31 days, with a tail reaching 365 days. |


### Missing data Findings

- **840,765 customers (84.08%)** have no missing values across the five
  nullable columns.
- **159,235 customers (15.92%)** have at least one missing value.
- Most incomplete records are missing only one of these fields.


These patterns do not establish that values are missing completely at
random. Missingness mechanisms remain unverified.

Dropping every incomplete row would remove almost 16% of customers, so
imputation seems like the better approch here.

### IQR

| Feature | Flagged observations | Percentage of non-null values |
|---|---:|---:|
| Annual income | 44,871 | 4.63% |
| Monthly data usage | 56,428 | 5.94% |
| Monthly charges | 9,740 | 0.97% |
| Total charges | 45,654 | 4.57% |

Monthly-charge flags include 379 values below the lower IQR bound and
9,361 above the upper bound.

The IQR rule is different from checking documented  ranges.

### Decisions for next 

- Retain all observations and preserve missing values for now.
- Do not automatically remove or cap IQR-flagged values.
- Maybe transformations for skewed features during model development.
- Fit imputation, scaling, and any learned clipping thresholds on training
  data only.


### Data split stratified

In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split


RANDOM_STATE = 42

customers = connection.sql("""
    SELECT customer_id, churn
    FROM raw_customers
    ORDER BY customer_id
""").df()

if customers.empty:
    raise ValueError("Cannot split an empty dataset.")

if customers["customer_id"].isna().any():
    raise ValueError("Customer IDs must not be missing.")

if customers["customer_id"].duplicated().any():
    raise ValueError("Customer IDs must be unique before splitting.")

if customers["churn"].isna().any() or not customers["churn"].isin([0, 1]).all():
    raise ValueError("Churn labels must be non-null binary values.")

train_customers, holdout_customers = train_test_split(
    customers,
    test_size=0.30,
    stratify=customers["churn"],
    random_state=RANDOM_STATE,
)

validation_customers, test_customers = train_test_split(
    holdout_customers,
    test_size=0.50,
    stratify=holdout_customers["churn"],
    random_state=RANDOM_STATE,
)

split_manifest = pd.concat(
    [
        train_customers.assign(partition="train"),
        validation_customers.assign(partition="validation"),
        test_customers.assign(partition="test"),
    ],
    ignore_index=True,
).sort_values("customer_id").reset_index(drop=True)

if (
    len(split_manifest) != len(customers)
    or split_manifest["customer_id"].duplicated().any()
    or not split_manifest["customer_id"].equals(customers["customer_id"])
):
    raise ValueError("Partitions must cover every customer exactly once.")

if set(split_manifest["partition"]) != {"train", "validation", "test"}:
    raise ValueError("Expected three nonempty partitions.")

split_summary = (
    split_manifest.groupby("partition")
    .agg(
        customers=("customer_id", "size"),
        churned_customers=("churn", "sum"),
        churn_rate=("churn", "mean"),
    )
    .reindex(["train", "validation", "test"])
)

split_summary["share_percentage"] = (
    100 * split_summary["customers"] / len(customers)
)
split_summary["churn_percentage"] = 100 * split_summary["churn_rate"]

display(
    split_summary.drop(columns="churn_rate").round(3)
)

,customers,churned_customers,share_percentage,churn_percentage
partition,,,,
train,700000,69459,70.0,9.923
validation,150000,14884,15.0,9.923
test,150000,14884,15.0,9.923


In [16]:
SPLIT_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "customer_split_seed42_v1.parquet"
)

SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)

if SPLIT_PATH.exists():
    saved_manifest = (
        pd.read_parquet(SPLIT_PATH)
        .sort_values("customer_id")
        .reset_index(drop=True)
    )

    if not saved_manifest.equals(split_manifest):
        raise ValueError(
            "Existing split differs from the current assignments or labels. "
            "Do not overwrite it silently. Investigate dataset or dependency "
            "changes before creating a new split version."
        )

    print(f"Existing split verified: {SPLIT_PATH}")
else:
    with SPLIT_PATH.open("xb") as file:
        split_manifest.to_parquet(file, index=False)

    print(f"Split saved: {SPLIT_PATH}")

Split saved: C:\Users\antek\Customer_Chunrn_End_to_End\data\processed\customer_split_seed42_v1.parquet


In [17]:
connection.register("customer_split_manifest", split_manifest)

connection.sql("""
    CREATE OR REPLACE TEMP VIEW train_customers AS
    SELECT customers.*
    FROM raw_customers AS customers
    INNER JOIN customer_split_manifest AS assignments
        ON customers.customer_id = assignments.customer_id
    WHERE assignments.partition = 'train'
""")

training_rows = connection.sql("""
    SELECT COUNT(*)
    FROM train_customers
""").fetchall()[0][0]

if training_rows != len(train_customers):
    raise ValueError("Training view row count does not match the split.")

print(f"Training EDA ready: {training_rows:,} customers.")

Training EDA ready: 700,000 customers.


### From now exploration only on training data

In [18]:
import pandas as pd
import plotly.express as px
from IPython.display import display


training_baseline = connection.sql("""
    SELECT AVG(churn)
    FROM train_customers
""").fetchall()[0][0]

if training_baseline is None:
    raise ValueError("Training data is empty.")

print(f"Training churn rate: {100 * training_baseline:.2f}%")


EDA_COLUMNS = {
    "contract",
    "payment_method",
    "paperless_billing",
    "customer_satisfaction",
    "num_complaints",
    "late_payments",
    "num_service_calls",
}


def compare_churn(column: str, title: str) -> pd.DataFrame:
    if column not in EDA_COLUMNS:
        raise ValueError(f"Unsupported comparison column: {column}")

    results = connection.sql(f"""
        WITH baseline AS (
            SELECT AVG(churn) AS churn_rate
            FROM train_customers
        ),
        grouped AS (
            SELECT
                "{column}" AS category,
                COUNT(*) AS customers,
                SUM(churn) AS churned_customers,
                AVG(churn) AS churn_rate
            FROM train_customers
            GROUP BY "{column}"
        )
        SELECT
            category,
            customers,
            churned_customers,
            100.0 * customers / SUM(customers) OVER ()
                AS customer_percentage,
            100.0 * grouped.churn_rate AS churn_percentage,
            grouped.churn_rate / NULLIF(baseline.churn_rate, 0)
                AS lift
        FROM grouped
        CROSS JOIN baseline
        ORDER BY category NULLS LAST
    """).df()

    results["group"] = results["category"].map(
        lambda value: "Missing" if pd.isna(value) else str(value)
    )

    print(f"\n{title}")
    display(
        results[
            [
                "group",
                "customers",
                "churned_customers",
                "customer_percentage",
                "churn_percentage",
                "lift",
            ]
        ].round(3)
    )

    figure = px.bar(
        results,
        x="group",
        y="churn_percentage",
        text="customers",
        custom_data=[
            "customers",
            "churned_customers",
            "customer_percentage",
            "lift",
        ],
        title=title,
        labels={
            "group": column,
            "churn_percentage": "Churn rate (%)",
        },
    )

    figure.update_traces(
        texttemplate="n=%{text:,}",
        textposition="outside",
        cliponaxis=False,
        hovertemplate=(
            "Group: %{x}"
            "<br>Churn rate: %{y:.2f}%"
            "<br>Customers: %{customdata[0]:,}"
            "<br>Churned: %{customdata[1]:,}"
            "<br>Training share: %{customdata[2]:.2f}%"
            "<br>Lift: %{customdata[3]:.2f}x"
            "<extra></extra>"
        ),
    )

    figure.add_hline(
        y=100 * training_baseline,
        line_dash="dash",
        annotation_text="Training baseline",
    )

    figure.update_xaxes(type="category")
    figure.update_yaxes(
        rangemode="tozero",
        range=[
            0,
            min(
                110,
                max(
                    results["churn_percentage"].max(),
                    100 * training_baseline,
                ) * 1.2 + 1,
            ),
        ],
    )
    figure.update_layout(
        showlegend=False,
        margin=dict(t=80),
    )
    figure.show()

    return results

Training churn rate: 9.92%


In [19]:
categorical_churn_results = {}

for column, title in [
    ("contract", "Churn by contract type"),
    ("payment_method", "Churn by payment method"),
    ("paperless_billing", "Churn by paperless billing"),
]:
    categorical_churn_results[column] = compare_churn(column, title)


Churn by contract type


,group,customers,churned_customers,customer_percentage,churn_percentage,lift
0,month_to_month,13904,3676.0,1.986,26.438,2.664
1,one_year,385537,48786.0,55.077,12.654,1.275
2,two_year,300559,16997.0,42.937,5.655,0.570



Churn by payment method


,group,customers,churned_customers,customer_percentage,churn_percentage,lift
0,bank_transfer,175107,17349.0,25.015,9.908,0.998
1,credit_card,244807,24253.0,34.972,9.907,0.998
2,electronic_check,175440,17533.0,25.063,9.994,1.007
3,mailed_check,104646,10324.0,14.949,9.866,0.994



Churn by paperless billing


,group,customers,churned_customers,customer_percentage,churn_percentage,lift
0,False,174777,17362.0,24.968,9.934,1.001
1,True,525223,52097.0,75.032,9.919,1.000


In [20]:
behavioral_churn_results = {}

for column, title in [
    ("customer_satisfaction", "Churn by satisfaction score"),
    ("num_complaints", "Churn by complaints — last year"),
    ("late_payments", "Churn by late payments — last 3 months"),
    ("num_service_calls", "Churn by service calls — last month"),
]:
    behavioral_churn_results[column] = compare_churn(column, title)


Churn by satisfaction score


,group,customers,churned_customers,customer_percentage,churn_percentage,lift
0,1.0,27507,4565.0,3.930,16.596,1.673
1,2.0,34572,5541.0,4.939,16.027,1.615
2,3.0,48162,7015.0,6.880,14.565,1.468
3,4.0,61508,6671.0,8.787,10.846,1.093
4,5.0,75440,7790.0,10.777,10.326,1.041
5,6.0,89237,8443.0,12.748,9.461,0.954
6,7.0,102476,8821.0,14.639,8.608,0.867
7,8.0,116383,9477.0,16.626,8.143,0.821
8,9.0,130795,9789.0,18.685,7.484,0.754
9,Missing,13920,1347.0,1.989,9.677,0.975



Churn by complaints — last year


,group,customers,churned_customers,customer_percentage,churn_percentage,lift
0,0.0,337317,27355.0,48.188,8.110,0.817
1,1.0,235561,24625.0,33.652,10.454,1.054
2,2.0,83022,11120.0,11.860,13.394,1.350
3,3.0,19253,3412.0,2.750,17.722,1.786
4,4.0,3410,715.0,0.487,20.968,2.113
5,5.0,466,120.0,0.067,25.751,2.595
6,6.0,57,18.0,0.008,31.579,3.182
7,7.0,8,4.0,0.001,50.000,5.039
8,Missing,20906,2090.0,2.987,9.997,1.007



Churn by late payments — last 3 months


,group,customers,churned_customers,customer_percentage,churn_percentage,lift
0,0,469047,42600.0,67.007,9.082,0.915
1,1,187706,20795.0,26.815,11.078,1.116
2,2,37684,5104.0,5.383,13.544,1.365
3,3,5031,846.0,0.719,16.816,1.695
4,4,488,104.0,0.070,21.311,2.148
5,5,44,10.0,0.006,22.727,2.290



Churn by service calls — last month


,group,customers,churned_customers,customer_percentage,churn_percentage,lift
0,0,143759,11078.0,20.537,7.706,0.777
1,1,208028,18271.0,29.718,8.783,0.885
2,2,165289,16381.0,23.613,9.911,0.999
3,3,98034,11159.0,14.005,11.383,1.147
4,4,48983,6297.0,6.998,12.855,1.296
5,5,21705,3265.0,3.101,15.043,1.516
6,6,8782,1718.0,1.255,19.563,1.972
7,7,3474,786.0,0.496,22.625,2.280
8,8,1246,310.0,0.178,24.880,2.507
9,9,468,124.0,0.067,26.496,2.670


In [21]:
from statistics import NormalDist

import pandas as pd
import plotly.express as px


def plot_churn_intervals(
    results: pd.DataFrame,
    title: str,
) -> pd.DataFrame:
    results = results.copy()

    n = results["customers"].astype(float)
    k = results["churned_customers"].astype(float)

    if (
        results.empty
        or n.isna().any()
        or k.isna().any()
        or (n <= 0).any()
        or (k < 0).any()
        or (k > n).any()
    ):
        raise ValueError("Expected nonempty groups with valid customer counts.")

    proportion = k / n
    z = NormalDist().inv_cdf(0.975)

    denominator = 1 + z**2 / n
    center = (proportion + z**2 / (2 * n)) / denominator
    half_width = (
        z
        * (proportion * (1 - proportion) / n + z**2 / (4 * n**2)) ** 0.5
        / denominator
    )

    lower = (center - half_width).clip(0, 1).where(k != 0, 0.0)
    upper = (center + half_width).clip(0, 1).where(k != n, 1.0)

    results["churn_percentage"] = 100 * proportion
    results["ci_lower"] = 100 * lower
    results["ci_upper"] = 100 * upper
    results["error_plus"] = (
        results["ci_upper"] - results["churn_percentage"]
    )
    results["error_minus"] = (
        results["churn_percentage"] - results["ci_lower"]
    )

    display(
        results[
            [
                "group",
                "customers",
                "churned_customers",
                "churn_percentage",
                "ci_lower",
                "ci_upper",
            ]
        ].round(2)
    )

    figure = px.scatter(
        results,
        x="group",
        y="churn_percentage",
        error_y="error_plus",
        error_y_minus="error_minus",
        hover_data={
            "customers": True,
            "churned_customers": True,
            "ci_lower": ":.2f",
            "ci_upper": ":.2f",
            "error_plus": False,
            "error_minus": False,
        },
        title=title,
        labels={
            "group": "Group",
            "churn_percentage": "Churn rate (%)",
        },
    )

    figure.update_traces(marker_size=9)
    figure.update_xaxes(
        type="category",
        categoryorder="array",
        categoryarray=results["group"].tolist(),
    )
    figure.update_yaxes(rangemode="tozero")
    figure.add_hline(
        y=100 * training_baseline,
        line_dash="dash",
        annotation_text="Training baseline",
    )
    figure.show()

    return results

In [22]:
service_call_intervals = plot_churn_intervals(
    behavioral_churn_results["num_service_calls"],
    "Churn by service calls — 95% Wilson intervals",
)

,group,customers,churned_customers,churn_percentage,ci_lower,ci_upper
0,0,143759,11078.0,7.71,7.57,7.84
1,1,208028,18271.0,8.78,8.66,8.91
2,2,165289,16381.0,9.91,9.77,10.06
3,3,98034,11159.0,11.38,11.19,11.58
4,4,48983,6297.0,12.86,12.56,13.15
5,5,21705,3265.0,15.04,14.57,15.52
6,6,8782,1718.0,19.56,18.75,20.41
7,7,3474,786.0,22.63,21.26,24.05
8,8,1246,310.0,24.88,22.56,27.35
9,9,468,124.0,26.50,22.70,30.67


In [23]:
churn_interval_results = {}

for column, results in {
    **categorical_churn_results,
    **behavioral_churn_results,
}.items():
    if column == "num_service_calls":
        continue

    churn_interval_results[column] = plot_churn_intervals(
        results,
        f"Churn by {column} — 95% Wilson intervals",
    )

,group,customers,churned_customers,churn_percentage,ci_lower,ci_upper
0,month_to_month,13904,3676.0,26.44,25.71,27.18
1,one_year,385537,48786.0,12.65,12.55,12.76
2,two_year,300559,16997.0,5.66,5.57,5.74


,group,customers,churned_customers,churn_percentage,ci_lower,ci_upper
0,bank_transfer,175107,17349.0,9.91,9.77,10.05
1,credit_card,244807,24253.0,9.91,9.79,10.03
2,electronic_check,175440,17533.0,9.99,9.85,10.13
3,mailed_check,104646,10324.0,9.87,9.69,10.05


,group,customers,churned_customers,churn_percentage,ci_lower,ci_upper
0,False,174777,17362.0,9.93,9.79,10.07
1,True,525223,52097.0,9.92,9.84,10.00


,group,customers,churned_customers,churn_percentage,ci_lower,ci_upper
0,1.0,27507,4565.0,16.60,16.16,17.04
1,2.0,34572,5541.0,16.03,15.64,16.42
2,3.0,48162,7015.0,14.57,14.25,14.88
3,4.0,61508,6671.0,10.85,10.60,11.09
4,5.0,75440,7790.0,10.33,10.11,10.55
5,6.0,89237,8443.0,9.46,9.27,9.66
6,7.0,102476,8821.0,8.61,8.44,8.78
7,8.0,116383,9477.0,8.14,7.99,8.30
8,9.0,130795,9789.0,7.48,7.34,7.63
9,Missing,13920,1347.0,9.68,9.20,10.18


,group,customers,churned_customers,churn_percentage,ci_lower,ci_upper
0,0.0,337317,27355.0,8.11,8.02,8.20
1,1.0,235561,24625.0,10.45,10.33,10.58
2,2.0,83022,11120.0,13.39,13.16,13.63
3,3.0,19253,3412.0,17.72,17.19,18.27
4,4.0,3410,715.0,20.97,19.63,22.37
5,5.0,466,120.0,25.75,21.99,29.91
6,6.0,57,18.0,31.58,21.00,44.48
7,7.0,8,4.0,50.00,21.52,78.48
8,Missing,20906,2090.0,10.00,9.60,10.41


,group,customers,churned_customers,churn_percentage,ci_lower,ci_upper
0,0,469047,42600.0,9.08,9.00,9.16
1,1,187706,20795.0,11.08,10.94,11.22
2,2,37684,5104.0,13.54,13.20,13.89
3,3,5031,846.0,16.82,15.81,17.87
4,4,488,104.0,21.31,17.91,25.16
5,5,44,10.0,22.73,12.84,36.99


## Training Churn Exploration

### Approach

For each group, I calculated:
- Customer count and churned customer count.
- Churn rate and share of the training population.
- Lift relative to the overall training churn rate of approximately 9.92%.
- Pointwise 95% Wilson confidence intervals.

### Contract Type

| Contract | Customers | Churn rate | Lift |
|---|---:|---:|---:|
| Month-to-month | 13,904 | 26.44% | 2.66 |
| One-year | 385,537 | 12.65% | 1.28 |
| Two-year | 300,559 | 5.66% | 0.57 |

Contract type shows a strong association with churn.

Although monthly contracts have the highest rate, one-year contracts
contain the most churned customers: 48,786. 

The observed monthly-contract churn rate is below the documentation's
claim of 40% or more.

### Payment Method and Paperless Billing

Payment method churn rates range from approximately 9.87% to 9.99%.
Paperless-billing groups both have churn rates near 9.93%.

Neither variable shows a significant association in these
comparisons. This does not prove that they have no conditional or
interaction-based predictive value. They will not be excluded  solely
based on these plots, however they are likely to be dropped during model building.

### Customer Satisfaction

Churn generally decreases as satisfaction increases:

- Score 1: **16.60%**
- Score 6: **9.46%**
- Score 9: **7.48%**
- Missing satisfaction: **9.68%**

The missing group is near the training baseline. Its similarity to score
6.

### Complaints

Churn increases with complaint count:

- Zero complaints: **8.11%**
- One complaint: **10.45%**
- Three complaints: **17.72%**
- Missing complaints: **10.00%**

Extreme complaint groups are sparse. For example, seven complaints
correspond to only eight customers, four of who churned.

The resulting 50% rate is highly uncertain and should not be treated as
a stable estimate.

### Late Payments

Churn increases from **9.08%** with no late payments to **21.31%** with
four and **22.73%** with five.

However, the four and five payment groups contain only 488 and 44
customers, respectively. Their estimates are less precise than those
for common payment counts.

### Service Calls

Churn broadly increases with service-call count, but decreases after
10 calls in the observed training sample:

| Calls | Customers | Churned | Churn rate | 95% Wilson interval |
|---|---:|---:|---:|---:|
| 10 | 157 | 51 | 32.48% | 25.65–40.15% |
| 11 | 51 | 15 | 29.41% | 18.71–43.00% |
| 12 | 24 | 4 | 16.67% | 6.68–35.85% |

The final groups are small and have wide uncertainty intervals.
At 12 calls, one additional churned customer would change the rate by about
4.17 percentage points.

The decrease does not provide strong evidence of a reversal.

### Interpreting Confidence Intervals

Wilson intervals help communicate uncertainty in group churn rates,
particularly for rare groups.


In [24]:
band_definitions = {
    "tenure": {
        "title": "Churn by tenure — 95% Wilson intervals",
        "expression": """
            CASE
                WHEN tenure < 6 THEN 1
                WHEN tenure <= 12 THEN 2
                WHEN tenure <= 24 THEN 3
                WHEN tenure <= 36 THEN 4
                WHEN tenure <= 60 THEN 5
                ELSE 6
            END
        """,
        "labels": {
            1: "1–5 months",
            2: "6–12 months",
            3: "13–24 months",
            4: "25–36 months",
            5: "37–60 months",
            6: "61–72 months",
        },
    },
    "monthlycharges": {
        "title": "Churn by monthly charges — 95% Wilson intervals",
        "expression": """
            CASE
                WHEN monthlycharges < 50 THEN 1
                WHEN monthlycharges < 75 THEN 2
                WHEN monthlycharges < 100 THEN 3
                WHEN monthlycharges < 125 THEN 4
                WHEN monthlycharges < 150 THEN 5
                WHEN monthlycharges <= 200 THEN 6
                ELSE 7
            END
        """,
        "labels": {
            1: "Below $50",
            2: "$50 to <75",
            3: "$75 to <100",
            4: "$100 to <125",
            5: "$125 to <150",
            6: "$150–200",
            7: "Above $200",
        },
    },
}

In [25]:
band_churn_results = {}

expected_training_rows = connection.sql("""
    SELECT COUNT(*)
    FROM train_customers
""").fetchall()[0][0]

for feature, definition in band_definitions.items():
    results = connection.sql(f"""
        WITH banded AS (
            SELECT
                {definition["expression"]} AS band_order,
                churn
            FROM train_customers
        ),
        grouped AS (
            SELECT
                band_order,
                COUNT(*) AS customers,
                SUM(churn) AS churned_customers,
                AVG(churn) AS churn_rate
            FROM banded
            GROUP BY band_order
        ),
        baseline AS (
            SELECT AVG(churn) AS churn_rate
            FROM train_customers
        )
        SELECT
            band_order,
            customers,
            churned_customers,
            100.0 * customers / SUM(customers) OVER ()
                AS customer_percentage,
            100.0 * grouped.churn_rate AS churn_percentage,
            grouped.churn_rate / NULLIF(baseline.churn_rate, 0)
                AS lift
        FROM grouped
        CROSS JOIN baseline
        ORDER BY band_order
    """).df()

    if results["customers"].sum() != expected_training_rows:
        raise ValueError(f"{feature}: bands do not cover all training rows.")

    results["group"] = results["band_order"].map(definition["labels"])

    if results["group"].isna().any():
        raise ValueError(f"{feature}: an unexpected band has no label.")

    print(f"\n{feature.upper()} — GROUP METRICS")

    display(
        results[
            [
                "group",
                "customers",
                "churned_customers",
                "customer_percentage",
                "churn_percentage",
                "lift",
            ]
        ].round(3)
    )

    band_churn_results[feature] = plot_churn_intervals(
        results,
        definition["title"],
    )


TENURE — GROUP METRICS


,group,customers,churned_customers,customer_percentage,churn_percentage,lift
0,1–5 months,154295,15948.0,22.042,10.336,1.042
1,6–12 months,138159,13933.0,19.737,10.085,1.016
2,13–24 months,160039,16014.0,22.863,10.006,1.008
3,25–36 months,97609,9555.0,13.944,9.789,0.987
4,37–60 months,94704,8912.0,13.529,9.410,0.948
5,61–72 months,55194,5097.0,7.885,9.235,0.931


,group,customers,churned_customers,churn_percentage,ci_lower,ci_upper
0,1–5 months,154295,15948.0,10.34,10.19,10.49
1,6–12 months,138159,13933.0,10.08,9.93,10.24
2,13–24 months,160039,16014.0,10.01,9.86,10.15
3,25–36 months,97609,9555.0,9.79,9.60,9.98
4,37–60 months,94704,8912.0,9.41,9.23,9.60
5,61–72 months,55194,5097.0,9.23,9.00,9.48



MONTHLYCHARGES — GROUP METRICS


,group,customers,churned_customers,customer_percentage,churn_percentage,lift
0,Below $50,39425,4471.0,5.632,11.341,1.143
1,$50 to <75,182860,18385.0,26.123,10.054,1.013
2,$75 to <100,296410,29824.0,42.344,10.062,1.014
3,$100 to <125,145920,13864.0,20.846,9.501,0.958
4,$125 to <150,30728,2525.0,4.390,8.217,0.828
5,$150–200,3976,321.0,0.568,8.073,0.814
6,Above $200,681,69.0,0.097,10.132,1.021


,group,customers,churned_customers,churn_percentage,ci_lower,ci_upper
0,Below $50,39425,4471.0,11.34,11.03,11.66
1,$50 to <75,182860,18385.0,10.05,9.92,10.19
2,$75 to <100,296410,29824.0,10.06,9.95,10.17
3,$100 to <125,145920,13864.0,9.50,9.35,9.65
4,$125 to <150,30728,2525.0,8.22,7.92,8.53
5,$150–200,3976,321.0,8.07,7.27,8.96
6,Above $200,681,69.0,10.13,8.08,12.63


### Tenure and Monthly Charges

- **Tenure:** Churn decreases from **10.34%** at 1–5 months to **9.23%**
  at 61–72 months. Narrow confidence intervals indicate precise because of large groups,
  but the association is modest arund **1.10 percentage points**.

- **Monthly charges:** Churn generally decreases across charge bands, from
  **11.34%** below $50 to **8.07%** at $150–200. Above $200, it rises to
  **10.13%**, but this group contains only **681 customers**, giving a wider
  95% confidence interval of **8.08–12.63%**.


In [26]:
missingness_columns = [
    "annual_income",
    "customer_satisfaction",
    "num_complaints",
    "avg_monthly_gb",
    "credit_score",
]

missingness_churn_results = {}

for column in missingness_columns:
    results = connection.sql(f"""
        WITH grouped AS (
            SELECT
                "{column}" IS NULL AS is_missing,
                COUNT(*) AS customers,
                SUM(churn) AS churned_customers,
                AVG(churn) AS churn_rate
            FROM train_customers
            GROUP BY is_missing
        ),
        baseline AS (
            SELECT AVG(churn) AS churn_rate
            FROM train_customers
        )
        SELECT
            CASE
                WHEN is_missing THEN 'Missing'
                ELSE 'Observed'
            END AS "group",
            customers,
            churned_customers,
            100.0 * customers / SUM(customers) OVER ()
                AS customer_percentage,
            100.0 * grouped.churn_rate AS churn_percentage,
            grouped.churn_rate / NULLIF(baseline.churn_rate, 0)
                AS lift
        FROM grouped
        CROSS JOIN baseline
        ORDER BY is_missing
    """).df()

    print(f"\n{column.upper()} — MISSINGNESS COMPARISON")

    display(
        results[
            [
                "group",
                "customers",
                "churned_customers",
                "customer_percentage",
                "churn_percentage",
                "lift",
            ]
        ].round(3)
    )

    missingness_churn_results[column] = plot_churn_intervals(
        results,
        f"{column}: missing vs observed — 95% Wilson intervals",
    )


ANNUAL_INCOME — MISSINGNESS COMPARISON


,group,customers,churned_customers,customer_percentage,churn_percentage,lift
0,Observed,679031,67388.0,97.004,9.924,1.000
1,Missing,20969,2071.0,2.996,9.876,0.995


,group,customers,churned_customers,churn_percentage,ci_lower,ci_upper
0,Observed,679031,67388.0,9.92,9.85,10.00
1,Missing,20969,2071.0,9.88,9.48,10.29



CUSTOMER_SATISFACTION — MISSINGNESS COMPARISON


,group,customers,churned_customers,customer_percentage,churn_percentage,lift
0,Observed,686080,68112.0,98.011,9.928,1.001
1,Missing,13920,1347.0,1.989,9.677,0.975


,group,customers,churned_customers,churn_percentage,ci_lower,ci_upper
0,Observed,686080,68112.0,9.93,9.86,10.00
1,Missing,13920,1347.0,9.68,9.20,10.18


... (5 outputs left out)

,group,customers,churned_customers,churn_percentage,ci_lower,ci_upper
0,Observed,665057,65902.0,9.91,9.84,9.98
1,Missing,34943,3557.0,10.18,9.87,10.50



CREDIT_SCORE — MISSINGNESS COMPARISON


,group,customers,churned_customers,customer_percentage,churn_percentage,lift
0,Observed,671583,66632.0,95.94,9.922,1.000
1,Missing,28417,2827.0,4.06,9.948,1.003


,group,customers,churned_customers,churn_percentage,ci_lower,ci_upper
0,Observed,671583,66632.0,9.92,9.85,9.99
1,Missing,28417,2827.0,9.95,9.61,10.30


### Missingness and Churn

Missing and observed groups have similar churn rates across the five
nullable features, suggesting little marginal association between
missingness and churn in the training data. This strongly implies missigness is not related to churn.


In [27]:
def explore_joint_groups(
    band_feature: str,
    segment_expression: str,
    segment_order: list[str],
    title: str,
) -> pd.DataFrame:
    definition = band_definitions[band_feature]

    results = connection.sql(f"""
        WITH grouped AS (
            SELECT
                {definition["expression"]} AS band_order,
                {segment_expression} AS segment,
                COUNT(*) AS customers,
                SUM(churn) AS churned_customers
            FROM train_customers
            GROUP BY band_order, segment
        )
        SELECT
            *,
            100.0 * churned_customers / customers AS churn_percentage
        FROM grouped
        ORDER BY segment, band_order
    """).df()

    expected_rows = connection.sql("""
        SELECT COUNT(*)
        FROM train_customers
    """).fetchall()[0][0]

    if results["customers"].sum() != expected_rows:
        raise ValueError("Joint groups do not cover all training customers.")

    results["group"] = results["band_order"].map(definition["labels"])

    if results["group"].isna().any():
        raise ValueError("Unexpected band without a label.")

    if set(results["segment"]) - set(segment_order):
        raise ValueError("Unexpected segment without a display order.")

    interval_results = []

    for segment in segment_order:
        segment_results = (
            results.loc[results["segment"] == segment]
            .sort_values("band_order")
            .copy()
        )

        if segment_results.empty:
            print(f"No training customers in segment: {segment}")
            continue

        print(f"\n{title} — {segment}")

        with_intervals = plot_churn_intervals(
            segment_results,
            f"{title} — {segment}",
        )
        interval_results.append(with_intervals)

    return pd.concat(interval_results, ignore_index=True)

In [28]:
tenure_contract_results = explore_joint_groups(
    band_feature="tenure",
    segment_expression="contract",
    segment_order=[
        "month_to_month",
        "one_year",
        "two_year",
    ],
    title="Churn by tenure within contract type",
)


Churn by tenure within contract type — month_to_month


,group,customers,churned_customers,churn_percentage,ci_lower,ci_upper
0,1–5 months,3017,1071.0,35.50,33.81,37.22
1,6–12 months,2695,642.0,23.82,22.25,25.47
2,13–24 months,3227,793.0,24.57,23.12,26.09
3,25–36 months,1975,489.0,24.76,22.91,26.71
4,37–60 months,1855,420.0,22.64,20.79,24.60
5,61–72 months,1135,261.0,23.00,20.64,25.53



Churn by tenure within contract type — one_year


,group,customers,churned_customers,churn_percentage,ci_lower,ci_upper
6,1–5 months,84872,11033.0,13.00,12.77,13.23
7,6–12 months,76015,9832.0,12.93,12.70,13.17
8,13–24 months,88217,11283.0,12.79,12.57,13.01
9,25–36 months,53852,6728.0,12.49,12.22,12.78
10,37–60 months,52374,6330.0,12.09,11.81,12.37
11,61–72 months,30207,3580.0,11.85,11.49,12.22



Churn by tenure within contract type — two_year


,group,customers,churned_customers,churn_percentage,ci_lower,ci_upper
12,1–5 months,66406,3844.0,5.79,5.61,5.97
13,6–12 months,59449,3459.0,5.82,5.63,6.01
14,13–24 months,68595,3938.0,5.74,5.57,5.92
15,25–36 months,41782,2338.0,5.60,5.38,5.82
16,37–60 months,40475,2162.0,5.34,5.13,5.56
17,61–72 months,23852,1256.0,5.27,4.99,5.56


In [29]:
charges_satisfaction_results = explore_joint_groups(
    band_feature="monthlycharges",
    segment_expression="""
        CASE
            WHEN customer_satisfaction IS NULL THEN 'Missing'
            WHEN customer_satisfaction <= 4 THEN 'Low (1-4)'
            WHEN customer_satisfaction <= 7 THEN 'Medium (5-7)'
            ELSE 'High (8-10)'
        END
    """,
    segment_order=[
        "Low (1-4)",
        "Medium (5-7)",
        "High (8-10)",
        "Missing",
    ],
    title="Churn by monthly charges within satisfaction group",
)


Churn by monthly charges within satisfaction group — Low (1-4)


,group,customers,churned_customers,churn_percentage,ci_lower,ci_upper
7,Below $50,9666,1399.0,14.47,13.79,15.19
8,$50 to <75,44725,5681.0,12.70,12.40,13.01
9,$75 to <100,72788,10520.0,14.45,14.20,14.71
10,$100 to <125,35733,5076.0,14.21,13.85,14.57
11,$125 to <150,7670,972.0,12.67,11.95,13.44
12,$150–200,1004,127.0,12.65,10.73,14.85
13,Above $200,163,17.0,10.43,6.61,16.07



Churn by monthly charges within satisfaction group — Medium (5-7)


,group,customers,churned_customers,churn_percentage,ci_lower,ci_upper
14,Below $50,15145,1696.0,11.20,10.71,11.71
15,$50 to <75,69902,6949.0,9.94,9.72,10.17
16,$75 to <100,113138,10603.0,9.37,9.20,9.54
17,$100 to <125,55724,4826.0,8.66,8.43,8.90
18,$125 to <150,11449,835.0,7.29,6.83,7.78
19,$150–200,1537,112.0,7.29,6.09,8.70
20,Above $200,258,33.0,12.79,9.25,17.42



Churn by monthly charges within satisfaction group — High (8-10)


,group,customers,churned_customers,churn_percentage,ci_lower,ci_upper
0,Below $50,13830,1284.0,9.28,8.81,9.78
1,$50 to <75,64723,5390.0,8.33,8.12,8.54
2,$75 to <100,104578,8148.0,7.79,7.63,7.96
3,$100 to <125,51463,3687.0,7.16,6.94,7.39
4,$125 to <150,10977,661.0,6.02,5.59,6.48
5,$150–200,1363,79.0,5.80,4.68,7.17
6,Above $200,244,17.0,6.97,4.40,10.87



Churn by monthly charges within satisfaction group — Missing


,group,customers,churned_customers,churn_percentage,ci_lower,ci_upper
21,Below $50,784,92.0,11.73,9.67,14.18
22,$50 to <75,3510,365.0,10.40,9.43,11.45
23,$75 to <100,5906,553.0,9.36,8.65,10.13
24,$100 to <125,3000,275.0,9.17,8.19,10.25
25,$125 to <150,632,57.0,9.02,7.03,11.51
26,$150–200,72,3.0,4.17,1.43,11.55
27,Above $200,16,2.0,12.50,3.50,36.02


## Joint-Group Exploration: Findings

### Tenure Within Contract Types

- **Month-to-month:** Customers with 1–5 months of tenure have **35.50%**
  churn (95% CI: **33.81–37.22%**), compared with approximately **23–25%**
  across most longer tenure bands.
- **One-year:** Churn gradually decreases from **13.00%** at 1–5 months
  to **11.85%** at 61–72 months.
- **Two-year:** Churn remains comparatively low, with a modest overall
  decrease from **5.79%** to **5.27%**.

The short tenure association is particularly pronounced for monthly
contracts. This suggests a candidate **short tenure × monthly contract**
interaction.

### Monthly Charges Within Satisfaction Groups

- **Low satisfaction (1–4):** Churn is generally around **12.7–14.5%**
  across charge bands up to $200, without a consistent decreasing pattern.
- **Medium satisfaction (5–7):** Churn generally decreases from **11.20%**
  below $50 to **7.29%** at $150–200.
- **High satisfaction (8–10):** Churn decreases from **9.28%**
  below $50 to **5.80%** at $150–200. Observed scores only reach 9.
- Across charge bands up to $200, lower satisfaction groups consistently
  have higher observed churn than higher satisfaction groups.

The charge with churn relationship differs across satisfaction groups,
suggesting a candidate interaction rather than one universal price effect.


### Modeling Implications

Retain tenure, contract, monthly charges, and satisfaction as candidate
inputs. Compare an additive baseline against selected interaction features
through training data cross validation, without changing raw values or
using validation/test data for exploratory feature selection.